In [2]:
import faiss
from langchain_text_splitters import RecursiveCharacterTextSplitter
import spacy

d:\Study\pyspider\GenAI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from sentence_transformers import SentenceTransformer

In [4]:
nlp = spacy.load('en_core_web_sm')

In [5]:
data = open('D:\Study\pyspider\GenAI\Embeddings\data.txt').read()
data

'====================================================================\nARTIFICIAL INTELLIGENCE (AI), MACHINE LEARNING (ML) AND\nDEEP LEARNING (DL) - A COMPLETE GUIDE\n====================================================================\n\nTable of Contents\n------------------\n1. Introduction\n2. What is Artificial Intelligence (AI)?\n3. History and Evolution of AI\n4. Types of Artificial Intelligence\n5. What is Machine Learning (ML)?\n6. How Machine Learning Works\n7. Types of Machine Learning\n   7.1 Supervised Learning\n   7.2 Unsupervised Learning\n   7.3 Semi-Supervised Learning\n   7.4 Reinforcement Learning\n8. Common Machine Learning Algorithms\n9. What is Deep Learning (DL)?\n10. How Deep Learning Works\n11. Neural Network Architectures\n12. Difference Between AI, ML and DL\n13. Applications of AI, ML and DL\n14. Popular Tools and Frameworks\n15. Challenges and Limitations\n16. Ethics and Responsible AI\n17. Future Trends\n18. Conclusion\n19. Glossary of Terms\n\n============

In [6]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 100,
    chunk_overlap = 20
)

In [7]:
chunks = splitter.split_text(data)
chunks

['====================================================================',
 'ARTIFICIAL INTELLIGENCE (AI), MACHINE LEARNING (ML) AND\nDEEP LEARNING (DL) - A COMPLETE GUIDE',
 '====================================================================',
 'Table of Contents\n------------------\n1. Introduction\n2. What is Artificial Intelligence (AI)?',
 '3. History and Evolution of AI\n4. Types of Artificial Intelligence',
 '5. What is Machine Learning (ML)?\n6. How Machine Learning Works\n7. Types of Machine Learning',
 '7.1 Supervised Learning\n   7.2 Unsupervised Learning\n   7.3 Semi-Supervised Learning',
 '7.4 Reinforcement Learning\n8. Common Machine Learning Algorithms\n9. What is Deep Learning (DL)?',
 '10. How Deep Learning Works\n11. Neural Network Architectures\n12. Difference Between AI, ML and DL',
 '13. Applications of AI, ML and DL\n14. Popular Tools and Frameworks\n15. Challenges and Limitations',
 '16. Ethics and Responsible AI\n17. Future Trends\n18. Conclusion\n19. Glossary o

In [8]:
embeddings_model = SentenceTransformer(
    model_name_or_path = 'sentence-transformers/all-miniLM-L6-V2'
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3608.11it/s]


In [9]:
embeddings = embeddings_model.encode(chunks).astype('float32')

In [10]:
embeddings

array([[-0.06323688,  0.03542084, -0.06788158, ..., -0.04697526,
         0.12158132,  0.00340066],
       [-0.06588248, -0.1015351 ,  0.04885887, ...,  0.0551771 ,
         0.02041178,  0.01357141],
       [-0.06323688,  0.03542084, -0.06788158, ..., -0.04697526,
         0.12158132,  0.00340066],
       ...,
       [-0.06089264, -0.04349817,  0.01146283, ...,  0.08068978,
        -0.04202182, -0.01862268],
       [ 0.03419521, -0.08500054,  0.02022358, ...,  0.09262124,
         0.10323888, -0.03602435],
       [-0.06641545, -0.04677554, -0.02388168, ...,  0.02338019,
        -0.05283674, -0.00145342]], shape=(424, 384), dtype=float32)

In [11]:
dimension = embeddings.shape[1]

In [12]:
faiss.normalize_L2(embeddings)

In [13]:
index = faiss.IndexFlatIP(dimension) # dot product 
# index = faiss.IndexFlatIP(dimension) # euclidean distance


In [14]:
index.add(embeddings) # stored embeddings inside faiss DB

In [15]:
Query = 'Explain Machine Learning ?'

In [16]:
query_embeddings = embeddings_model.encode(Query).astype('float32')

In [17]:
query_embeddings = query_embeddings.reshape(1, -1)

In [18]:
faiss.normalize_L2(query_embeddings)

In [19]:
'''
1st - value -> distance
2nd value -> index (where the chunks is present)
'''
index.search(query_embeddings, k=4) # search_documents

(array([[0.77072  , 0.7176594, 0.7077872, 0.6845054]], dtype=float32),
 array([[115,  23, 117, 413]]))

In [20]:
# user defined fun rag_query(query)
# return distance, index

In [26]:
def rag_query(query):
    query_embeddings = embeddings_model.encode(query).astype('float32')
    query_embeddings = query_embeddings.reshape(1, -1)
    faiss.normalize_L2(query_embeddings)
    distance, indeces =index.search(query_embeddings, k=6)# search_documents
    return distance, indeces
query = 'Explain Machine Learning ?'
rag_query(query)

(array([[0.77072   , 0.7176594 , 0.7077872 , 0.6845054 , 0.68054074,
         0.6800113 ]], dtype=float32),
 array([[115,  23, 117, 413, 133, 296]]))